# Training Run Review

Review the persisted smoke-bundle artifacts for one baseline training and evaluation run.
The notebook stays inspection-only: it reads the frozen JSON and Markdown artifacts, reconstructs
typed metadata models, and uses plotting only for human review.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from motifml.evaluation.reporting import render_qualitative_report_markdown
from motifml.training.contracts import EvaluationRunMetadata, TrainingRunMetadata
from motifml.training.training_loop import TrainingEpochMetrics, TrainingHistory

pd.set_option("display.max_colwidth", 120)


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the MotifML repository root.")


repo_root = _find_repo_root(Path.cwd())
smoke_bundle_root = Path(
    os.environ.get(
        "MOTIFML_TRAINING_SMOKE_BUNDLE_ROOT",
        repo_root / "tests" / "fixtures" / "training" / "smoke_bundle",
    )
).resolve()


def load_json(path: Path) -> object:
    return json.loads(path.read_text(encoding="utf-8"))


training_history_payload = load_json(smoke_bundle_root / "training_history.json")
metrics_payload = load_json(smoke_bundle_root / "metrics.json")
samples_payload = load_json(
    smoke_bundle_root / "evaluation" / "qualitative_samples.json"
)
qualitative_report_text = (smoke_bundle_root / "qualitative_report.md").read_text(
    encoding="utf-8"
)
training_run_metadata = TrainingRunMetadata.from_json_dict(
    load_json(smoke_bundle_root / "training_run_metadata.json")
)
evaluation_run_metadata = EvaluationRunMetadata.from_json_dict(
    load_json(smoke_bundle_root / "evaluation_run_metadata.json")
)
training_history = TrainingHistory(
    epochs=tuple(
        TrainingEpochMetrics(**epoch_payload)
        for epoch_payload in training_history_payload["epochs"]
    ),
    best_epoch_index=int(training_history_payload["best_epoch_index"]),
    best_validation_loss=float(training_history_payload["best_validation_loss"]),
)
rendered_report = render_qualitative_report_markdown(
    evaluation_run_id=metrics_payload["evaluation_run_id"],
    training_run_id=metrics_payload["training_run_id"],
    split_metrics=metrics_payload["splits"],
    samples_by_split=samples_payload["samples"],
)
report_match = rendered_report == qualitative_report_text

display(
    Markdown(
        "## Training Run Overview\n\n"
        f"- Training Run: `{training_run_metadata.training_run_id}`\n"
        f"- Evaluation Run: `{evaluation_run_metadata.evaluation_run_id}`\n"
        f"- Device: `{training_run_metadata.device}`\n"
        f"- Seed: `{training_run_metadata.seed}`\n"
        f"- Best Epoch Index: `{training_history.best_epoch_index}`\n"
        f"- Evaluated Splits: `{', '.join(split.value for split in evaluation_run_metadata.evaluated_splits)}`\n"
        f"- Report Match: `{report_match}`\n"
    )
)

In [ ]:
history_frame = pd.DataFrame(
    [
        {
            "epoch_index": epoch.epoch_index,
            "train_loss": epoch.train_loss,
            "validation_loss": epoch.validation_loss,
            "learning_rate": epoch.learning_rate,
        }
        for epoch in training_history.epochs
    ]
)
display(
    Markdown(
        "## Training Curves\n\n"
        f"- Best Validation Loss: `{training_history.best_validation_loss:.6f}`\n"
        f"- Epoch Count: `{len(training_history.epochs)}`\n"
    )
)
loss_figure = px.line(
    history_frame,
    x="epoch_index",
    y=["train_loss", "validation_loss"],
    markers=True,
    title="Loss Curves",
)
display(loss_figure)

In [ ]:
evaluation_rows = [
    {
        "split": split_name,
        "perplexity": split_payload["quantitative"]["perplexity"],
        "accuracy": split_payload["quantitative"]["accuracy"],
        "top_k_accuracy": split_payload["quantitative"]["top_k_accuracy"],
    }
    for split_name, split_payload in sorted(metrics_payload["splits"].items())
]
evaluation_frame = pd.DataFrame(evaluation_rows)
display(
    Markdown(
        "## Evaluation Curves\n\n- Metrics are loaded directly from `metrics.json`."
    )
)
evaluation_figure = px.line(
    evaluation_frame.melt(id_vars="split", var_name="metric", value_name="value"),
    x="split",
    y="value",
    color="metric",
    markers=True,
    title="Perplexity, Accuracy, and Top-k Accuracy",
)
display(evaluation_figure)

In [ ]:
sample_rows = [
    {
        "split": split_name,
        "relative_path": sample["relative_path"],
        "prompt": sample["prompt_summary"],
        "reference": sample["reference_summary"],
        "generated": sample["generated_summary"],
    }
    for split_name, split_samples in sorted(samples_payload["samples"].items())
    for sample in split_samples
]
display(
    Markdown(
        "## Qualitative Review\n\n"
        f"- Sample Count: `{len(sample_rows)}`\n"
        f"- First Sample Document: `{sample_rows[0]['relative_path']}`\n"
    )
)
display(pd.DataFrame(sample_rows))
display(Markdown(qualitative_report_text))